# nSTATPaperExamples

This notebook is a Python-native tutorial derived from the MATLAB workflow name, implemented from scratch for `nSTAT-python`.

- Execution group: `smoke`
- Workflow family: `decoding_1d`
- Paper DOI: `10.1016/j.jneumeth.2012.08.009`
- PMID: `22981419`
- Help page: `docs/help/examples/nSTATPaperExamples.md`


Notebook source link: [nSTATPaperExamples.ipynb](https://github.com/cajigaslab/nSTAT-python/blob/main/notebooks/nSTATPaperExamples.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "nSTATPaperExamples"
FAMILY = "decoding_1d"
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"nSTATPaperExamples: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"nSTATPaperExamples: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"nSTATPaperExamples: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"nSTATPaperExamples: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# nSTATPaperExamples: multi-section paper-style workflow summary.
import json
from pathlib import Path
from scipy.io import loadmat
from nstat.compat.matlab import Analysis, DecodingAlgorithms


def resolve_repo_root() -> Path:
    candidates = [Path.cwd().resolve()]
    candidates.append(candidates[0].parent)
    candidates.append(candidates[1].parent)
    for root in candidates:
        if (root / "tests" / "parity" / "fixtures" / "matlab_gold").exists():
            return root
    return candidates[0]


repo_root = resolve_repo_root()
fixture_root = repo_root / "tests" / "parity" / "fixtures" / "matlab_gold"

# Section 1 (MATLAB paper examples): Poisson GLM fit proxy from gold fixture.
m_pp = loadmat(fixture_root / "PPSimExample_gold.mat")
X_pp = np.asarray(m_pp["X"], dtype=float)
y_pp = np.asarray(m_pp["y"], dtype=float).reshape(-1)
dt_pp = float(np.asarray(m_pp["dt"], dtype=float).reshape(-1)[0])
b_pp = np.asarray(m_pp["b"], dtype=float).reshape(-1)
expected_rate_pp = np.asarray(m_pp["expected_rate"], dtype=float).reshape(-1)

fit_pp = Analysis.fitGLM(X=X_pp, y=y_pp, fitType="poisson", dt=dt_pp)
rate_hat_pp = np.asarray(fit_pp.predict(X_pp), dtype=float).reshape(-1)
coef_pp = np.concatenate([[float(fit_pp.intercept)], np.asarray(fit_pp.coefficients, dtype=float)])
coef_err_pp = float(np.linalg.norm(coef_pp - b_pp))
rate_rel_err_pp = float(
    np.mean(np.abs(rate_hat_pp - expected_rate_pp) / np.maximum(np.abs(expected_rate_pp), 1e-12))
)

# Section 2 (MATLAB decoding example with history): posterior + MAP path parity.
m_dec = loadmat(fixture_root / "DecodingExampleWithHist_gold.mat")
spike_counts = np.asarray(m_dec["spike_counts"], dtype=float)
tuning = np.asarray(m_dec["tuning"], dtype=float)
transition = np.asarray(m_dec["transition"], dtype=float)
expected_decoded = np.asarray(m_dec["expected_decoded"], dtype=int).reshape(-1)
expected_post = np.asarray(m_dec["expected_posterior"], dtype=float)

decoded_hist, posterior_hist = DecodingAlgorithms.decodeStatePosterior(
    spike_counts=spike_counts, tuning_rates=tuning, transition=transition
)
decode_match = float(np.mean(decoded_hist == expected_decoded))
posterior_max_abs = float(np.max(np.abs(posterior_hist - expected_post)))

# Section 3 (MATLAB hippocampal place-cell example): weighted-center decode parity.
m_pc = loadmat(fixture_root / "HippocampalPlaceCellExample_gold.mat")
spike_counts_pc = np.asarray(m_pc["spike_counts_pc"], dtype=float)
tuning_curves = np.asarray(m_pc["tuning_curves"], dtype=float)
expected_weighted = np.asarray(m_pc["expected_decoded_weighted"], dtype=float).reshape(-1)

decoded_weighted = DecodingAlgorithms.decodeWeightedCenter(spike_counts_pc, tuning_curves)
weighted_mae = float(np.mean(np.abs(decoded_weighted - expected_weighted)))
weighted_max_err = float(np.max(np.abs(decoded_weighted - expected_weighted)))

# Section 4 (MATLAB PSTH/trial-significance): CI + significance matrix parity.
m_psth = loadmat(fixture_root / "PSTHEstimation_gold.mat")
spike_matrix_psth = np.asarray(m_psth["spike_matrix_psth"], dtype=float)
alpha_psth = float(np.asarray(m_psth["alpha_psth"], dtype=float).reshape(-1)[0])
expected_rate_psth = np.asarray(m_psth["expected_rate_psth"], dtype=float).reshape(-1)
expected_prob_psth = np.asarray(m_psth["expected_prob_psth"], dtype=float)
expected_sig_psth = np.asarray(m_psth["expected_sig_psth"], dtype=int)

rate_psth, prob_psth, sig_psth = DecodingAlgorithms.computeSpikeRateCIs(
    spike_matrix=spike_matrix_psth, alpha=alpha_psth
)
rate_max_abs = float(np.max(np.abs(rate_psth - expected_rate_psth)))
prob_max_abs = float(np.max(np.abs(prob_psth - expected_prob_psth)))
sig_mismatch = int(np.sum(np.abs(sig_psth - expected_sig_psth)))

# Section 5: audit metadata from MATLAB gold export.
audit_path = fixture_root / "nSTATPaperExamples_audit_gold.json"
audit = json.loads(audit_path.read_text(encoding="utf-8"))
audit_alignment = str(audit.get("alignment_status", ""))
audit_code_lines = int(audit.get("matlab_code_lines", 0))
audit_ref_images = int(audit.get("matlab_reference_image_count", 0))

fig, axes = plt.subplots(2, 3, figsize=(13.0, 8.4))
axes[0, 0].plot(expected_rate_pp[:1200], "k", linewidth=1.0, label="MATLAB gold")
axes[0, 0].plot(rate_hat_pp[:1200], "tab:blue", linewidth=1.0, label="Python fit")
axes[0, 0].set_title("Paper Exp 1 proxy: GLM rate fit")
axes[0, 0].legend(loc="upper right", fontsize=8)

axes[0, 1].plot(expected_decoded[:180], "k", linewidth=1.0, label="MATLAB decoded")
axes[0, 1].plot(decoded_hist[:180], "tab:green", linewidth=0.9, label="Python decoded")
axes[0, 1].set_title("Paper Exp 5 proxy: decoding path")
axes[0, 1].legend(loc="upper right", fontsize=8)

im0 = axes[0, 2].imshow(np.abs(posterior_hist - expected_post), aspect="auto", origin="lower", cmap="magma")
axes[0, 2].set_title("Posterior absolute error")
fig.colorbar(im0, ax=axes[0, 2], fraction=0.045, pad=0.02)

axes[1, 0].plot(expected_weighted, "k", linewidth=1.0, label="MATLAB weighted")
axes[1, 0].plot(decoded_weighted, "tab:red", linewidth=0.9, label="Python weighted")
axes[1, 0].set_title("Paper Exp 4 proxy: weighted decode")
axes[1, 0].legend(loc="upper right", fontsize=8)

field = tuning_curves[6].reshape(5, 8)
im1 = axes[1, 1].imshow(field, origin="lower", cmap="jet", aspect="auto")
axes[1, 1].set_title("Example place field (unit 7)")
fig.colorbar(im1, ax=axes[1, 1], fraction=0.045, pad=0.02)

im2 = axes[1, 2].imshow(prob_psth, origin="lower", cmap="gray_r", aspect="auto")
yy, xx = np.where(sig_psth > 0)
if xx.size:
    axes[1, 2].plot(xx, yy, "r*", markersize=3)
axes[1, 2].set_title("Trial significance matrix")
fig.colorbar(im2, ax=axes[1, 2], fraction=0.045, pad=0.02)
plt.tight_layout()
plt.show()

assert coef_err_pp < 0.7
assert rate_rel_err_pp < 0.30
assert decode_match >= 1.0
assert posterior_max_abs < 1e-9
assert weighted_mae < 1e-10
assert weighted_max_err < 1e-10
assert rate_max_abs < 1e-10
assert prob_max_abs < 1e-10
assert sig_mismatch == 0
assert audit_alignment == "validated"
assert audit_code_lines > 1000

CHECKPOINT_METRICS = {
    "coef_error_pp": float(coef_err_pp),
    "rate_rel_err_pp": float(rate_rel_err_pp),
    "decode_match": float(decode_match),
    "weighted_mae": float(weighted_mae),
    "psth_rate_max_abs": float(rate_max_abs),
    "sig_mismatch": float(sig_mismatch),
    "matlab_code_lines": float(audit_code_lines),
    "matlab_ref_images": float(audit_ref_images),
}
CHECKPOINT_LIMITS = {
    "coef_error_pp": (0.0, 0.7),
    "rate_rel_err_pp": (0.0, 0.30),
    "decode_match": (1.0, 1.0),
    "weighted_mae": (0.0, 1e-10),
    "psth_rate_max_abs": (0.0, 1e-10),
    "sig_mismatch": (0.0, 0.0),
    "matlab_code_lines": (1000.0, 5000.0),
    "matlab_ref_images": (1.0, 1000.0),
}


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


## Next steps

- Compare this notebook with the corresponding MATLAB helpfile output in the validation PDF.
- Use `tools/reports/generate_validation_pdf.py` to run side-by-side parity scoring.
- Refine model assumptions for this specific example until parity thresholds pass.
